https://dataframely.readthedocs.io/stable/guides/quickstart.html

In [1]:
!pip install dataframely


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import dataframely as dy
import polars as pl


class HouseSchema(dy.Schema):
    zip_code = dy.String(nullable=False, min_length=6)
    num_bedrooms = dy.UInt8(nullable=False)
    num_bathrooms = dy.UInt8(nullable=False)
    price = dy.Float64(nullable=False)

    @dy.rule()
    def reasonable_bathroom_to_bedroom_ratio(cls) -> pl.Expr:
        ratio = pl.col("num_bathrooms") / pl.col("num_bedrooms")
        return (ratio >= 1 / 3) & (ratio <= 3)


In [3]:
import polars as pl

df = pl.DataFrame({
    "zip_code": ["01234", "01234", "1", "213", "123", "213"],
    "num_bedrooms": [2, 2, 1, None, None, 2],
    "num_bathrooms": [1, 2, 1, 1, 0, 8],
    "price": [100_000, 110_000, 50_000, 80_000, 60_000, 160_000]
})

# Validate the data and cast columns to expected types
try:
    validated_df: dy.DataFrame[HouseSchema] = HouseSchema.validate(df, cast=True)
except Exception as e:
    print(e)


3 rules failed validation:
 - 'reasonable_bathroom_to_bedroom_ratio' failed for 1 rows
 * Column 'zip_code' failed validation for 1 rules:
   - 'min_length' failed for 6 rows
 * Column 'num_bedrooms' failed validation for 1 rules:
   - 'nullability' failed for 2 rows


In [4]:
# Filter the data and cast columns to expected types
good, failure = HouseSchema.filter(df, cast=True)

# Inspect the reasons for the failed rows
print(failure.counts())

{'reasonable_bathroom_to_bedroom_ratio': 1, 'zip_code|min_length': 6, 'num_bedrooms|nullability': 2}


In [5]:
failed_df = failure.invalid()
failed_df

zip_code,num_bedrooms,num_bathrooms,price
str,i64,i64,i64
"""01234""",2,1,100000
"""01234""",2,2,110000
"""1""",1,1,50000
"""213""",null,1,80000
"""123""",null,0,60000
"""213""",2,8,160000


In [6]:
# Inspect the co-occurrences of validation failures
failure.cooccurrence_counts()

{frozenset({'reasonable_bathroom_to_bedroom_ratio', 'zip_code|min_length'}): 1,
 frozenset({'num_bedrooms|nullability', 'zip_code|min_length'}): 2,
 frozenset({'zip_code|min_length'}): 3}

In [7]:
import polars as pl 

file_path: str = "/home/user/py-pannguyen-learn/data/raw/sales_data_sample.xlsx"
df = pl.read_excel(file_path)
df.head()

Order_ID,Date,Category,Region,Customer_Segment,Payment_Method,Quantity,Unit_Price,Discount_Rate,Customer_Rating,Returned,Subcategory,Total_Price,Discount_Amount,Net_Sales,Shipping_Cost,Profit_Margin,Profit,Month
str,date,str,str,str,str,i64,f64,f64,i64,bool,str,f64,f64,f64,f64,f64,f64,str
"""ORD-10000""",2024-07-12,"""Books""","""East""","""Business""","""PayPal""",1,316.58,0.2,5,false,"""Comics""",316.58,63.316,253.264,0.0,0.33,83.57712,"""2024-07"""
"""ORD-10001""",2025-03-15,"""Sports""","""West""","""Premium""","""Bank Transfer""",5,279.76,0.1,null,true,"""Water Sports""",1398.8,139.88,1258.92,0.0,0.2,251.784,"""2025-03"""
"""ORD-10002""",2024-12-27,"""Home & Kitchen""","""Central""","""Business""","""Bank Transfer""",6,209.61,0.05,4,false,"""Furniture""",1257.66,62.883,1194.777,0.0,0.37,442.06749,"""2024-12"""
"""ORD-10003""",2024-07-16,"""Home & Kitchen""","""North""","""Regular""","""Credit Card""",3,265.47,0.15,4,false,"""Kitchenware""",796.41,119.4615,676.9485,0.0,0.23,155.698155,"""2024-07"""
"""ORD-10004""",2024-06-11,"""Electronics""","""North""","""New""","""Bank Transfer""",9,449.32,0.0,4,false,"""Accessories""",4043.88,0.0,4043.88,0.0,0.37,1496.2356,"""2024-06"""


In [8]:
origin_cols: list = df.columns
def normailzed_columns(columns: pl.DataFrame.columns) -> list:
    return [col.strip().lower() for col in columns]
normalized_cols: list = normailzed_columns(origin_cols)
normalized_cols

['order_id',
 'date',
 'category',
 'region',
 'customer_segment',
 'payment_method',
 'quantity',
 'unit_price',
 'discount_rate',
 'customer_rating',
 'returned',
 'subcategory',
 'total_price',
 'discount_amount',
 'net_sales',
 'shipping_cost',
 'profit_margin',
 'profit',
 'month']

In [9]:
df.columns = normalized_cols
df.head()

order_id,date,category,region,customer_segment,payment_method,quantity,unit_price,discount_rate,customer_rating,returned,subcategory,total_price,discount_amount,net_sales,shipping_cost,profit_margin,profit,month
str,date,str,str,str,str,i64,f64,f64,i64,bool,str,f64,f64,f64,f64,f64,f64,str
"""ORD-10000""",2024-07-12,"""Books""","""East""","""Business""","""PayPal""",1,316.58,0.2,5,false,"""Comics""",316.58,63.316,253.264,0.0,0.33,83.57712,"""2024-07"""
"""ORD-10001""",2025-03-15,"""Sports""","""West""","""Premium""","""Bank Transfer""",5,279.76,0.1,null,true,"""Water Sports""",1398.8,139.88,1258.92,0.0,0.2,251.784,"""2025-03"""
"""ORD-10002""",2024-12-27,"""Home & Kitchen""","""Central""","""Business""","""Bank Transfer""",6,209.61,0.05,4,false,"""Furniture""",1257.66,62.883,1194.777,0.0,0.37,442.06749,"""2024-12"""
"""ORD-10003""",2024-07-16,"""Home & Kitchen""","""North""","""Regular""","""Credit Card""",3,265.47,0.15,4,false,"""Kitchenware""",796.41,119.4615,676.9485,0.0,0.23,155.698155,"""2024-07"""
"""ORD-10004""",2024-06-11,"""Electronics""","""North""","""New""","""Bank Transfer""",9,449.32,0.0,4,false,"""Accessories""",4043.88,0.0,4043.88,0.0,0.37,1496.2356,"""2024-06"""


In [37]:
from typing import Literal
def check_datetime_format(column: str, dt_format: str, dt_type: Literal["date", "datetime"]) -> pl.Series:
    if dt_type not in ("date", "datetime"):
        raise ValueError("Type must be either 'date' or 'datetime'")

    str_series = pl.col(column).str
    if dt_type == "date":
        return str_series.to_date(format=dt_format, strict=False)
    else:
        return str_series.to_datetime(format=dt_format, strict=False)

In [38]:
def check_datetime_range(
    col: str,
    dt_min: str | datetime.datetime,
    dt_max: str | datetime.datetime,
    dt_format: str,
    dt_type: Literal["date", "datetime"]
) -> pl.Series:
    dt_format_check = check_datetime_format(col, dt_format, dt_type)
    return (
        (dt_format_check.is_not_null())
        & (dt_format_check >= dt_min)
        & (dt_format_check <= dt_max)
    )

In [40]:
import dataframely as dy
import datetime

class SaleSchema(dy.Schema):
    order_id = dy.String(nullable=False, unique=True)
    customer_rating = dy.Int64(
        nullable=False,
        min=0,
        max=5
    )
    month = dy.String(
        nullable=False,
    )
    @dy.rule()
    def validate_month_format_and_range(cls) -> pl.Expr:
        return check_datetime_range(
            "month",
            datetime.date(2000, 1, 1),
            datetime.date.today(),
            "%Y-%m",
            "date"
        )

# --- PIPELINE IMPLEMENTATION ---

# 1. Execute soft validation
valid_df, failure_info = SaleSchema.filter(df, cast=True)

if failure_info.counts():
    print("❌ CRITICAL: Data quality gate triggered.")

    # 🚀 PRODUCTION FIX: Extract complete original data via Anti-Join
    # This filters the raw 'df', keeping rows where 'order_id' does NOT exist in 'valid_df'
    invalid_raw_df = df.join(valid_df, on="order_id", how="anti")

    # 2. Export the complete bad data for audit trailing
    invalid_raw_df.write_csv("invalid_sales_records.csv")
    print(
        f"Isolated {len(invalid_raw_df)} complete raw rows to invalid_sales_records.csv"
    )

    # 3. Fail loudly to halt downstream ingestion
    raise ValueError(
        f"DataQualityError: Ingestion halted. Clean rows: {len(valid_df)}, Isolated rows: {len(invalid_raw_df)}"
    )

❌ CRITICAL: Data quality gate triggered.
Isolated 55 complete raw rows to invalid_sales_records.csv


ValueError: DataQualityError: Ingestion halted. Clean rows: 945, Isolated rows: 55